# Eight-signal BB84 teaching workflow

Keep the source seed-3 signal inputs and 2048 actual observations. Prepare with
native Python qfunc, read `qba` by name, then reconcile bases with the same inputs.
This small demonstration retains the source modal-packet selection and pairwise
parity illustration; it does not establish cryptographic security.


In [ ]:
import flagquantum as fq
from flagquantum.drawer import draw
from examples.synthesis.community.paper_implementation_project.bb84.bb84_implementation.main import (
    ALICE_BASES,
    BOB_BASES,
    MESSAGE,
    build_program,
)
from examples.synthesis.community.paper_implementation_project.bb84.bb84_implementation.overview.workflow import (
    run_workflow,
)

print("Actual message:", MESSAGE)
print("Actual Alice bases:", ALICE_BASES)
print("Actual Bob bases:", BOB_BASES)
program = build_program()
print(program.report["resources"])
print(draw(program.circuit, show_all_qubits=True))


## Native quantum readout

The public named readout returns numeric words. The workflow presents observations
in logical signal order without asking the user to decode bitstrings or wires.


In [ ]:
result = fq.run(program)
probabilities = dict(program.probabilities(result, registers=("qba",)))
print("Observed support probabilities:", probabilities)
print("Probability mass:", sum(probabilities.values()))


## Actual observations

Draw 2048 final computational-basis observations from the executed native
program, with explicit local sampling seed 3. Keep the most observed eight-signal
word as in the source; resolve count ties by ascending numeric word.


In [ ]:
workflow = run_workflow(
    message=MESSAGE,
    alice_bases=ALICE_BASES,
    bob_bases=BOB_BASES,
    shots=2048,
    seed=3,
)
print("Actual observations:", sum(row.count for row in workflow.observations))
print("Most observed packet:", workflow.selected)
print("First five observed rows:", workflow.observations[:5])


## Basis reconciliation and teaching postprocessing

The source omitted the preparation inputs at its last function call and generated
unrelated random inputs instead. This native version explicitly passes the actual
quantum inputs. Keep the original 0.15 error threshold and adjacent-pair XOR.
The matching-bit filter and parity pairs are illustrative, not cryptographic
error correction or privacy amplification. Missing matches and threshold
rejections stay explicit; they never fabricate a key.


In [ ]:
report = workflow.report
print("Matching signal positions:", report.matching_positions)
print("Alice raw key:", report.alice_raw_key)
print("Bob raw key:", report.bob_raw_key)
print("Error rate:", report.error_rate)
print("Matching-bit teaching filter:", report.corrected_key)
print("Illustrative parity pairs:", report.final_key)
